# CS3081 Assignment: Uncertainty and Bayesian Networks — Exercise 5(c)

**Alaa Alqurashi / S25100435**

## 1. Build the network

In [1]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TQDM_DISABLE"] = "1"

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling

model = DiscreteBayesianNetwork([("heatwave", "overload"), ("heatwave", "powercut"),
                                 ("overload", "powercut"), ("powercut", "cancel")])

cpd_h = TabularCPD("heatwave", 2, [[0.3], [0.7]], state_names={"heatwave": ["yes", "no"]})

cpd_a = TabularCPD("overload", 2, [[0.6, 0.1], [0.4, 0.9]],
                   evidence=["heatwave"], evidence_card=[2],
                   state_names={"overload": ["yes", "no"], "heatwave": ["yes", "no"]})

# columns: (H=yes,A=yes), (H=yes,A=no), (H=no,A=yes), (H=no,A=no)
cpd_c = TabularCPD("powercut", 2, [[0.7, 0.2, 0.5, 0.05], [0.3, 0.8, 0.5, 0.95]],
                   evidence=["heatwave", "overload"], evidence_card=[2, 2],
                   state_names={"powercut": ["yes", "no"], "heatwave": ["yes", "no"], "overload": ["yes", "no"]})

cpd_l = TabularCPD("cancel", 2, [[0.8, 0.1], [0.2, 0.9]],
                   evidence=["powercut"], evidence_card=[2],
                   state_names={"cancel": ["yes", "no"], "powercut": ["yes", "no"]})

model.add_cpds(cpd_h, cpd_a, cpd_c, cpd_l)
print(model.check_model())

True


In [2]:
print(cpd_c)

+---------------+---------------+---------------+---------------+--------------+
| heatwave      | heatwave(yes) | heatwave(yes) | heatwave(no)  | heatwave(no) |
+---------------+---------------+---------------+---------------+--------------+
| overload      | overload(yes) | overload(no)  | overload(yes) | overload(no) |
+---------------+---------------+---------------+---------------+--------------+
| powercut(yes) | 0.7           | 0.2           | 0.5           | 0.05         |
+---------------+---------------+---------------+---------------+--------------+
| powercut(no)  | 0.3           | 0.8           | 0.5           | 0.95         |
+---------------+---------------+---------------+---------------+--------------+


## 2. Exact inference with Variable Elimination

In [3]:
infer = VariableElimination(model)
result = infer.query(["heatwave"], evidence={"cancel": "yes"})
print(result)
print("P(heatwave=yes | cancel=yes) =", round(result.get_value(heatwave="yes"), 4))

+---------------+-----------------+
| heatwave      |   phi(heatwave) |
+===============+=================+
| heatwave(yes) |          0.5367 |
+---------------+-----------------+
| heatwave(no)  |          0.4633 |
+---------------+-----------------+
P(heatwave=yes | cancel=yes) = 0.5367


## 3. Rejection sampling with 10,000 samples

In [4]:
sampler = BayesianModelSampling(model)
samples = sampler.rejection_sample(evidence=[State("cancel", "yes")], size=10000, seed=42, show_progress=False)
print(samples.head())
print("Number of samples:", len(samples))
estimate = (samples["heatwave"] == "yes").mean()
print("Estimated P(heatwave=yes | cancel=yes) =", round(estimate, 4))

  heatwave overload powercut cancel
0      yes      yes      yes    yes
1      yes      yes      yes    yes
2      yes       no      yes    yes
3       no       no       no    yes
4      yes      yes      yes    yes
Number of samples: 10000
Estimated P(heatwave=yes | cancel=yes) = 0.543


## 4. Comparison

The exact answer from Variable Elimination is P(heatwave = yes | cancel = yes) = 0.5367, and rejection sampling with 10,000 samples gave an estimate of 0.5430, which is very close but not exactly the same. The small difference (0.0063) comes from random sampling, and it would usually get smaller if we used more samples.